# 01. Ingestion Pipeline

**Тема 7: Retrieval-Augmented Generation**
Курс «Промышленная разработка AI-агентов» · ФКН ВШЭ

---

В этом ноутбуке мы строим **ingestion pipeline** — первую половину RAG-системы.
На выходе получим готовую базу знаний: векторный индекс (Chroma) и лексический индекс (BM25).

В качестве учебного корпуса используем открытые русскоязычные источники про искусственный интеллект.
Загрузим их несколькими способами, чтобы увидеть, как ingestion pipeline работает с разными типами данных:

1. PDF-документ через `PyPDFLoader`;
2. обычную веб-страницу через `WebBaseLoader`;
3. статьи русской Википедии через MediaWiki API.

### Что разберём:
1. Загрузка и очистка документов из разных источников
2. Три стратегии чанкинга — сравниваем на одном тексте
3. Эмбеддинги через `EmbeddingsGigaR`
4. Векторный индекс в Chroma с метаданными
5. BM25-индекс для лексического поиска
6. Сохранение индексов на диск

> **Следующий ноутбук** (`02_retrieval_generation.ipynb`) использует готовую базу знаний из этого.


## Источники корпуса

В базу знаний попадают тексты из следующих открытых источников:

| Тип | Источник | Как используется |
|---|---|---|
| PDF | [Кодекс этики в сфере разработки и применения искусственного интеллекта на финансовом рынке](https://www.cbr.ru/content/document/file/178667/code_09072025.pdf) | постраничная загрузка через `PyPDFLoader` |
| Web | [Искусственный интеллект на финансовом рынке](https://www.cbr.ru/fintech/primenenie-iskusstvennogo-intellekta-na-finansovom-rynke/) | загрузка HTML-страницы через `WebBaseLoader` |
| Wiki | [Искусственный интеллект](https://ru.wikipedia.org/wiki/Искусственный_интеллект) | загрузка текста статьи через MediaWiki API |
| Wiki | [Искусственный интеллект в проектах Викимедиа](https://ru.wikipedia.org/wiki/Искусственный_интеллект_в_проектах_Викимедиа) | загрузка текста статьи через MediaWiki API |

Сами тексты не хранятся в репозитории: ноутбук скачивает их при запуске и сохраняет в индексы только обработанные чанки с метаданными.


In [ ]:
!python --version

In [ ]:
# Установка зависимостей (запустить один раз в выбранном Jupyter kernel)
# %pip install -r ../requirements.txt

# После установки зависимостей перезапустите kernel: Kernel -> Restart.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv

# Ищем корень репозитория по .env.example, чтобы ноутбук одинаково работал
# и при запуске из корня репо, и при запуске из папки topic7_rag.
REPO_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / ".env.example").exists()),
    Path.cwd(),
)
ENV_PATH = REPO_ROOT / ".env"

if ENV_PATH.exists():
    load_dotenv(ENV_PATH)
    env_status = str(ENV_PATH)
else:
    env_status = f"не найден ({REPO_ROOT / '.env.example'} найден)"

GIGACHAT_CREDENTIALS = os.getenv("GIGACHAT_CREDENTIALS")
GIGACHAT_SCOPE = os.getenv("GIGACHAT_SCOPE", "GIGACHAT_API_PERS")
GIGACHAT_MODEL = os.getenv("GIGACHAT_MODEL", "GigaChat-2-Max")
GIGACHAT_EMBEDDINGS = os.getenv("GIGACHAT_EMBEDDINGS_MODEL", "EmbeddingsGigaR")
SOURCE_FETCH_TIMEOUT = int(os.getenv("SOURCE_FETCH_TIMEOUT", "30"))

# Путь из .env может быть относительным. Делаем его стабильным относительно папки topic7_rag,
# потому что именно туда ноутбуки сохраняют учебные индексы и рабочие артефакты.
NOTEBOOK_DIR = REPO_ROOT / "topic7_rag"
_chroma_dir = Path(os.getenv("CHROMA_PERSIST_DIR", "./data/chroma_db"))
CHROMA_PERSIST_DIR = str(_chroma_dir if _chroma_dir.is_absolute() else NOTEBOOK_DIR / _chroma_dir)

print("✓ Конфигурация загружена")
print(f"  Env file:             {env_status}")
print(f"  LLM:                  {GIGACHAT_MODEL}")
print(f"  Embeddings:           {GIGACHAT_EMBEDDINGS}")
print(f"  Chroma dir:           {CHROMA_PERSIST_DIR}")
print(f"  Source fetch timeout: {SOURCE_FETCH_TIMEOUT}s")
print(f"  GigaChat credentials: {'заданы' if GIGACHAT_CREDENTIALS else 'не заданы'}")


## 1. Загрузка документов

LangChain предоставляет загрузчики для большинства форматов: PDF, HTML, Markdown, Word и др.
В этом ноутбуке мы соберём небольшой русскоязычный корпус из открытых источников.

| Канал | Как грузим | Зачем в демо |
|---|---|---|
| PDF | скачиваем во временную папку → `PyPDFLoader` | показывает работу с постраничными PDF-документами |
| Web | `WebBaseLoader` | показывает загрузку обычной HTML-страницы |
| Wiki | MediaWiki API → `Document` | показывает контролируемую загрузку энциклопедических текстов |

Для каждого источника сохраним URL, тип источника и лицензионную заметку в метаданных, чтобы дальше можно было строить citations и attribution.


In [ ]:
from __future__ import annotations

import json
import os
import tempfile
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

USER_AGENT = "IndustrialAIAgentsRAG/1.0 (+https://github.com/educational-rag-demo)"
os.environ.setdefault("USER_AGENT", USER_AGENT)

from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_core.documents import Document

RAG_SOURCES = [
    {
        "source_id": "cbr_ai_ethics_pdf",
        "source_type": "pdf",
        "title": "Кодекс этики в сфере разработки и применения искусственного интеллекта на финансовом рынке",
        "url": "https://www.cbr.ru/content/document/file/178667/code_09072025.pdf",
        "filename": "cbr_ai_ethics_code_2025.pdf",
        "license_note": "Публичный документ Банка России; перед распространением проверьте условия использования материалов сайта ЦБ.",
    },
    {
        "source_id": "cbr_ai_finmarket_web",
        "source_type": "web",
        "title": "Искусственный интеллект на финансовом рынке",
        "url": "https://www.cbr.ru/fintech/primenenie-iskusstvennogo-intellekta-na-finansovom-rynke/",
        "filename": "cbr_ai_finmarket.html",
        "license_note": "Публичная страница Банка России; перед распространением проверьте условия использования материалов сайта ЦБ.",
    },
    {
        "source_id": "wiki_artificial_intelligence",
        "source_type": "wiki",
        "title": "Искусственный интеллект",
        "wiki_title": "Искусственный интеллект",
        "filename": "wiki_iskusstvennyy_intellekt.txt",
        "license_note": "Wikipedia text: CC BY-SA; сохраняйте attribution и ссылку на страницу.",
    },
    {
        "source_id": "wiki_ai_in_wikimedia",
        "source_type": "wiki",
        "title": "Искусственный интеллект в проектах Викимедиа",
        "wiki_title": "Искусственный интеллект в проектах Викимедиа",
        "filename": "wiki_ai_in_wikimedia.txt",
        "license_note": "Wikipedia text: CC BY-SA; сохраняйте attribution и ссылку на страницу.",
    },
]


def fetch_bytes(url: str) -> bytes:
    """Скачивает источник без записи в репозиторий."""
    request = Request(url, headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=SOURCE_FETCH_TIMEOUT) as response:
        return response.read()


def attach_source_metadata(docs: list[Document], source: dict) -> list[Document]:
    """Приводит метаданные разных загрузчиков к одному виду."""
    normalized = []
    for doc in docs:
        metadata = dict(doc.metadata)
        metadata.update({
            "source_id": source["source_id"],
            "source_type": source["source_type"],
            "title": source["title"],
            "source_url": source.get("url"),
            "filename": source["filename"],
            "license_note": source["license_note"],
        })
        # LangChain loaders часто кладут source сами; оставим URL как канонический source.
        metadata["source"] = source.get("url") or metadata.get("source") or source["source_id"]
        normalized.append(Document(page_content=doc.page_content, metadata=metadata))
    return normalized


def load_pdf_source(source: dict, tmp_dir: Path) -> list[Document]:
    pdf_path = tmp_dir / source["filename"]
    pdf_path.write_bytes(fetch_bytes(source["url"]))
    docs = PyPDFLoader(str(pdf_path), mode="page").load()
    return attach_source_metadata(docs, source)


def load_web_source(source: dict) -> list[Document]:
    loader = WebBaseLoader(
        web_paths=[source["url"]],
        header_template={"User-Agent": USER_AGENT},
        requests_kwargs={"timeout": SOURCE_FETCH_TIMEOUT},
        raise_for_status=True,
    )
    docs = loader.load()
    return attach_source_metadata(docs, source)


def load_wiki_source(source: dict) -> list[Document]:
    params = urlencode({
        "action": "query",
        "prop": "extracts|info",
        "explaintext": 1,
        "redirects": 1,
        "inprop": "url",
        "format": "json",
        "titles": source["wiki_title"],
    })
    api_url = f"https://ru.wikipedia.org/w/api.php?{params}"
    payload = json.loads(fetch_bytes(api_url).decode("utf-8"))
    pages = payload["query"]["pages"]
    page = next(iter(pages.values()))
    if "missing" in page:
        raise ValueError(f"Wikipedia page not found: {source['wiki_title']}")

    url = page.get("fullurl") or f"https://ru.wikipedia.org/wiki/{source['wiki_title'].replace(' ', '_')}"
    enriched_source = {**source, "url": url}
    doc = Document(
        page_content=page.get("extract", ""),
        metadata={
            "page": 0,
            "wiki_pageid": page.get("pageid"),
            "wiki_title": page.get("title"),
        },
    )
    return attach_source_metadata([doc], enriched_source)


def load_source(source: dict, tmp_dir: Path) -> list[Document]:
    if source["source_type"] == "pdf":
        return load_pdf_source(source, tmp_dir)
    if source["source_type"] == "web":
        return load_web_source(source)
    if source["source_type"] == "wiki":
        return load_wiki_source(source)
    raise ValueError(f"Unknown source_type: {source['source_type']}")


raw_docs = []
failures = []
with tempfile.TemporaryDirectory(prefix="rag_sources_") as tmp:
    tmp_dir = Path(tmp)
    for source in RAG_SOURCES:
        try:
            loaded = load_source(source, tmp_dir)
            raw_docs.extend(loaded)
            print(f"✓ {source['source_type']:>4} | {source['source_id']}: {len(loaded)} document/page objects")
        except (HTTPError, URLError, TimeoutError, ValueError, Exception) as exc:
            failures.append((source, exc))
            print(f"✗ {source['source_id']}: {type(exc).__name__}: {exc}")

if failures:
    print("\nНекоторые источники не загрузились. Для занятия можно повторить ячейку или временно убрать источник из RAG_SOURCES.")

if not raw_docs:
    raise RuntimeError("Не удалось загрузить ни одного документа. Проверьте интернет-доступ и SOURCE_FETCH_TIMEOUT.")

print(f"\nВсего загружено Document/page objects: {len(raw_docs)}")
print("Источники:")
for source_id in sorted({doc.metadata["source_id"] for doc in raw_docs}):
    docs_for_source = [doc for doc in raw_docs if doc.metadata["source_id"] == source_id]
    meta = docs_for_source[0].metadata
    print(f"  - {source_id} ({meta['source_type']}): {len(docs_for_source)} objects — {meta['title']}")


### Очистка текста

Перед индексацией полезно убрать лишние пробелы, спецсимволы и нормализовать текст.
Объём очистки зависит от источника: PDF обычно требует больше нормализации, чем HTML или wiki text.

`MIN_CHUNK_LEN` ниже используется не как "жёсткий фильтр полезных документов", а как защита от мусора:
номеров страниц, пустых HTML-блоков и коротких навигационных фрагментов.


In [ ]:
import re


def clean_text(text: str) -> str:
    """Лёгкая нормализация текста из PDF, HTML и wiki."""
    # Убираем переносы слов через дефис в конце строки
    text = re.sub(r"-\n(\w)", r"\1", text)
    # Объединяем строки внутри одного абзаца (не между абзацами)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    # Нормализуем множественные переносы строк
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Убираем лишние пробелы и неразрывные пробелы
    text = text.replace("\xa0", " ")
    text = re.sub(r" {2,}", " ", text)
    # Убираем пробелы в начале строк
    text = re.sub(r"^\s+", "", text, flags=re.MULTILINE)
    return text.strip()


MIN_CHUNK_LEN = 120  # отсекаем номера страниц, меню и прочий короткий шум

# Применяем очистку ко всем загруженным объектам
_docs = []
for doc in raw_docs:
    cleaned = clean_text(doc.page_content)
    if len(cleaned) >= MIN_CHUNK_LEN:
        _docs.append(Document(page_content=cleaned, metadata=doc.metadata))

docs = _docs

if not docs:
    raise RuntimeError("После очистки не осталось документов. Уменьшите MIN_CHUNK_LEN или проверьте загрузчики.")

print(f"✓ Очистка завершена: {len(docs)} непустых объектов (мин. {MIN_CHUNK_LEN} символов)")
print("\nПример фрагмента из первого объекта:\n")
print(docs[0].page_content[:400])


## 2. Стратегии чанкинга

Разобьём один документ тремя способами и сравним результаты.
Это поможет понять, как выбор стратегии влияет на границы фрагментов.


### 2.1 Фиксированный размер — `CharacterTextSplitter`

Самый простой способ: делит по разделителю (по умолчанию `\n\n`),
если чанк всё равно слишком велик — режет по символам без учёта структуры.

> Важно: после `clean_text()` в тексте может почти не остаться `\n\n`, поэтому на некоторых документах
> такой сплиттер неожиданно даёт один большой чанк. Это не баг, а ограничение простого правила разбиения.


In [ ]:
from langchain_text_splitters import CharacterTextSplitter

splitter_fixed = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
)

chunks_fixed = splitter_fixed.split_documents([docs[0]])

print(f"CharacterTextSplitter: {len(chunks_fixed)} чанков")
print(f"Средний размер: {sum(len(c.page_content) for c in chunks_fixed) // len(chunks_fixed)} символов")
print(f"\n--- Чанк #1 ---\n{chunks_fixed[0].page_content[:300]}")


### 2.2 Рекурсивное разбиение — `RecursiveCharacterTextSplitter`

**Рекомендуемый default** для большинства задач.
Алгоритм пробует разделить по `\n\n`, затем по `\n`, затем по `. `, затем посимвольно —
выбирает наиболее крупный разделитель, который ещё даёт чанк нужного размера.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter_recursive = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

chunks_recursive = splitter_recursive.split_documents([docs[0]])

print(f"RecursiveCharacterTextSplitter: {len(chunks_recursive)} чанков")
print(f"Средний размер: {sum(len(c.page_content) for c in chunks_recursive) // len(chunks_recursive)} символов")
print(f"\n--- Чанк #1 ---\n{chunks_recursive[0].page_content[:300]}")


### 2.3 Разбиение по предложениям

Гарантирует, что предложения никогда не разрезаются пополам.
Используем `RecursiveCharacterTextSplitter` с разделителями на уровне предложений.


In [ ]:
splitter_sentence = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=0,                        # overlap=0 при разбиении по предложениям
    separators=[". ", "! ", "? ", "\n"],   # только на границах предложений
    length_function=len,
)

chunks_sentence = splitter_sentence.split_documents([docs[0]])

print(f"Sentence-based: {len(chunks_sentence)} чанков")
print(f"Средний размер: {sum(len(c.page_content) for c in chunks_sentence) // len(chunks_sentence)} символов")
print(f"\n--- Чанк #1 ---\n{chunks_sentence[0].page_content[:300]}")


### 2.4 Сравнение стратегий


In [ ]:
import pandas as pd

def describe_chunks(name, chunks):
    sizes = [len(c.page_content) for c in chunks]
    return {
        "Стратегия": name,
        "Кол-во чанков": len(chunks),
        "Мин. размер": min(sizes),
        "Макс. размер": max(sizes),
        "Средний размер": round(sum(sizes) / len(sizes)),
    }

comparison = pd.DataFrame([
    describe_chunks("CharacterTextSplitter", chunks_fixed),
    describe_chunks("RecursiveCharacterTextSplitter", chunks_recursive),
    describe_chunks("Sentence-based", chunks_sentence),
])

print(comparison.to_string(index=False))


**Вывод:** `RecursiveCharacterTextSplitter` — хороший старт для большинства задач.
`CharacterTextSplitter` полезен как простой baseline, а sentence-based подход даёт аккуратные границы без разрезания предложений.

Для продакшна обычно начинают с `RecursiveCharacterTextSplitter`, а затем валидируют качество retrieval на тестовых запросах.


### Инициализация модели эмбеддингов

Инициализируем модель эмбеддингов GigaChat, которую будем использовать для индексации
и последующего векторного поиска в Chroma.


### Как заменить модель эмбеддингов на open-source

По умолчанию в ноутбуке используется `GigaChatEmbeddings`, потому что дальше в курсе мы работаем с GigaChat.
Но для ingestion pipeline это не принципиально: Chroma ожидает любой объект embeddings с методами `embed_documents()` и `embed_query()`.

Если хочется полностью локальный вариант без API-ключей, можно заменить ячейку ниже на open-source модель из `sentence-transformers`.
Например, для русскоязычного учебного корпуса подойдёт компактная мультиязычная модель:

- `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` — лёгкая, быстро работает на CPU;
- более тяжёлые multilingual/e5-модели могут дать лучшее качество, но будут медленнее и потребуют больше памяти.

Важно: если вы уже построили Chroma-индекс на одной модели эмбеддингов, при смене модели индекс нужно пересоздать заново.
Нельзя индексировать документы одной моделью, а искать запросы другой: векторы окажутся в разных пространствах.


In [ ]:
# Опциональный вариант: локальная open-source модель эмбеддингов
#
# 1. Установите дополнительные пакеты, если их нет в окружении:
# %pip install -U sentence-transformers langchain-huggingface
#
# 2. Перезапустите kernel.
#
# 3. Замените ячейку с GigaChatEmbeddings на код ниже.
#
# from langchain_huggingface import HuggingFaceEmbeddings
#
# embeddings = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
#     model_kwargs={"device": "cpu"},
#     encode_kwargs={"normalize_embeddings": True},
# )
#
# print("✓ Open-source модель эмбеддингов инициализирована")
# print("  Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


In [ ]:
from langchain_gigachat import GigaChatEmbeddings

embeddings = GigaChatEmbeddings(
    credentials=GIGACHAT_CREDENTIALS,
    scope=GIGACHAT_SCOPE,
    model=GIGACHAT_EMBEDDINGS,
    verify_ssl_certs=False,
)

print("✓ Модель эмбеддингов инициализирована")
print(f"  Model: {GIGACHAT_EMBEDDINGS}")


## 3. Обогащение метаданных

Метаданные — ключ к точной фильтрации при поиске.
Добавим к каждому чанку: `source_id`, тип источника, человекочитаемое имя файла/страницы,
стабильный `chunk_id`, номер страницы/объекта, номер чанка и лицензионную заметку.

> Поле `topic` в этом ноутбуке — это константный namespace-тег для демо-коллекции.
> В реальном ingestion pipeline такое поле должно нести переменную информацию: рубрику, дату, автора, источник и т.д.


In [ ]:
# Используем рекурсивный сплиттер как основной для всей базы знаний
# (RecursiveCharacterTextSplitter уже импортирован выше из langchain_text_splitters)
from collections import defaultdict
import hashlib

all_docs = splitter_recursive.split_documents(docs)

# Стабильный chunk_id нужен, чтобы повторный прогон ingestion не плодил дубли в Chroma.
# Делаем id детерминированным: source_id + page + индекс чанка внутри страницы/объекта + хеш текста.
chunk_ids = []
chunk_counters = defaultdict(int)

for doc in all_docs:
    source_id = doc.metadata.get("source_id", "unknown")
    page = doc.metadata.get("page", 0)
    chunk_index_in_doc = chunk_counters[(source_id, page)]
    chunk_counters[(source_id, page)] += 1

    content_hash = hashlib.sha1(doc.page_content.encode("utf-8")).hexdigest()[:12]
    chunk_id = f"{source_id}:p{page}:c{chunk_index_in_doc}:{content_hash}"
    doc_id = f"{source_id}:p{page}"

    doc.metadata.update({
        "doc_id": doc_id,
        "chunk_id": chunk_id,
        "chunk_index": chunk_index_in_doc,
        "chunk_size": len(doc.page_content),
        "topic": "ru_ai_open_corpus",
    })
    chunk_ids.append(chunk_id)

print(f"Всего чанков: {len(all_docs)}")
print(f"Уникальных chunk_id: {len(set(chunk_ids))}")
print("\nМетаданные первого чанка:")
for k, v in all_docs[0].metadata.items():
    print(f"  {k}: {v}")


## 4. Векторный индекс — Chroma

Создаём векторное хранилище с персистентностью на диск.
`langchain-chroma` интегрируется с LangChain и поддерживает метаданные-фильтры.
Используем `chromadb.PersistentClient` из документации Chroma и передаём стабильные `ids`,
чтобы повторный запуск ingestion не создавал дубли тех же чанков.


In [ ]:
import chromadb
from langchain_chroma import Chroma

chroma_client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)

# Полностью пересоздаём коллекцию, чтобы не накапливались чанки от прошлых запусков
# (например, если менялась стратегия фильтрации, чанкинга или состав онлайн-источников).
try:
    chroma_client.delete_collection("course_rag")
except Exception:
    pass
chroma_client.create_collection("course_rag")

vectorstore = Chroma(
    client=chroma_client,
    collection_name="course_rag",
    embedding_function=embeddings,
)

vectorstore.add_documents(documents=all_docs, ids=chunk_ids)

print(f"✓ Chroma индекс создан")
print(f"  Директория: {CHROMA_PERSIST_DIR}")
print(f"  Документов в индексе: {chroma_client.get_collection('course_rag').count()}")


### Быстрая проверка — векторный поиск

`similarity_search_with_score()` в Chroma возвращает не "оценку качества ответа", а distance-like score.
Для Chroma его нужно читать как расстояние: чем меньше значение, тем ближе документ к запросу.

Документация LangChain:
https://reference.langchain.com/python/langchain-chroma/vectorstores/Chroma/similarity_search_with_score


In [ ]:
test_query = "Какие принципы применения искусственного интеллекта перечисляет кодекс Банка России?"

results = vectorstore.similarity_search_with_score(test_query, k=3)

print(f"Запрос: «{test_query}»\n")
for i, (doc, score) in enumerate(results, 1):
    print(f"--- Результат {i} (score={score:.1f}) ---")
    print(
        f"Источник: {doc.metadata['filename']} | "
        f"тип={doc.metadata.get('source_type')} | "
        f"стр./obj={doc.metadata.get('page', '?')} | "
        f"id={doc.metadata.get('chunk_id')}"
    )
    print(f"Текст: {doc.page_content[:250]}...")
    print()


### Поиск с фильтрацией по метаданным


In [ ]:
# Фильтрация по конкретному источнику.
# Показывает как ограничить поиск одним документом/каналом загрузки, не меняя запрос.
target_source_id = "cbr_ai_finmarket_web"

results_filtered = vectorstore.similarity_search(
    "экспериментальный правовой режим цифровые инновации искусственный интеллект",
    k=3,
    filter={"source_id": target_source_id},
)

print(f"Результаты с фильтром по source_id='{target_source_id}':\n")
for i, doc in enumerate(results_filtered, 1):
    print(f"--- {i} (стр./obj={doc.metadata.get('page', '?')}, id={doc.metadata.get('chunk_id')}) ---")
    print(doc.page_content[:250])
    print()


## 5. Лексический индекс — BM25

BM25 — классический алгоритм ранжирования, основанный на статистике слов.
Он незаменим там, где нужно найти **точные совпадения**: термины, имена, артикулы, коды.

Семантический поиск может не найти «BM25» по запросу про «алгоритм ранжирования» —
а BM25-поиск найдёт точно, потому что ищет по буквам.


In [ ]:
from langchain_community.retrievers import BM25Retriever

# BM25Retriever строится из тех же документов, что и векторный индекс
bm25_retriever = BM25Retriever.from_documents(
    all_docs,
    k=3,
)

print("✓ BM25 индекс создан")
print(f"  Документов: {len(all_docs)}")


In [ ]:
# Демонстрируем преимущество BM25 на точных терминах.
# Запрос содержит юридически и технически точные формулировки — BM25 часто находит их надёжнее,
# чем семантический поиск, который может «уплыть» к похожим по смыслу текстам.
query_exact = "экспериментальный правовой режим цифровые инновации искусственный интеллект робототехника"

vector_results = vectorstore.similarity_search(query_exact, k=3)
bm25_results = bm25_retriever.invoke(query_exact)

print(f"Запрос: «{query_exact}»\n")

print("=== Векторный поиск ===")
for i, doc in enumerate(vector_results, 1):
    print(f"{i}. [{doc.metadata['filename']} стр./obj={doc.metadata.get('page','?')}] {doc.page_content[:180]}...")

print("\n=== BM25 поиск ===")
for i, doc in enumerate(bm25_results, 1):
    print(f"{i}. [{doc.metadata['filename']} стр./obj={doc.metadata.get('page','?')}] {doc.page_content[:180]}...")


**Обратите внимание:** на точном запросе с техническими терминами BM25 часто находит более релевантные результаты.
В следующем ноутбуке объединим оба метода в гибридный поиск через `EnsembleRetriever`.


## 6. Сохранение индексов

Chroma автоматически сохраняет данные на диск при указании `persist_directory`.
BM25 не имеет встроенного персистирования — сохраним объект через `pickle`.


In [ ]:
import pickle

# Chroma уже сохранена — просто убедимся
print(f"✓ Chroma сохранена в: {CHROMA_PERSIST_DIR}")
print(f"  Файлы: {list(Path(CHROMA_PERSIST_DIR).glob('*')) if Path(CHROMA_PERSIST_DIR).exists() else 'ещё не создана'}")

# BM25 сохраняем через pickle
bm25_path = Path(CHROMA_PERSIST_DIR).parent / "bm25_index.pkl"
bm25_path.parent.mkdir(parents=True, exist_ok=True)

with open(bm25_path, "wb") as f:
    pickle.dump(bm25_retriever, f)

print(f"\n✓ BM25 сохранён в: {bm25_path}")


### Загрузка существующих индексов (для следующих ноутбуков)


In [18]:
# Так будем загружать индексы в следующих ноутбуках:

# Chroma
chroma_client_loaded = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
chroma_collection_loaded = chroma_client_loaded.get_or_create_collection("course_rag")

vectorstore_loaded = Chroma(
    client=chroma_client_loaded,
    embedding_function=embeddings,
    collection_name="course_rag",
)

# BM25
with open(bm25_path, "rb") as f:
    bm25_retriever_loaded = pickle.load(f)

print("✓ Индексы загружены из файлов")
print(f"  Документов в Chroma: {chroma_collection_loaded.count()}")


✓ Индексы загружены из файлов
  Документов в Chroma: 350


## Итоги

В этом ноутбуке мы:

| Шаг | Что сделали |
|-----|-------------|
| Загрузка | Собрали корпус из PDF, HTML-страницы и русской Википедии |
| Очистка | Убрали переносы строк, лишние пробелы, артефакты PDF/HTML |
| Чанкинг | Сравнили 3 стратегии, выбрали `RecursiveCharacterTextSplitter` |
| Метаданные | Добавили `source_id`, `source_type`, `source_url`, `license_note`, стабильный `chunk_id`, `filename`, `chunk_index`, `chunk_size`, `topic` |
| Векторный индекс | Chroma с `EmbeddingsGigaR`, поиск + фильтрация по метаданным |
| Лексический индекс | BM25Retriever, сравнили с векторным поиском |
| Персистирование | Chroma на диск, BM25 через pickle |

**Следующий ноутбук:** `02_retrieval_generation.ipynb` — используем готовую базу знаний
для RAG 1.0, современного RAG на тулах и гибридного поиска (`EnsembleRetriever`).
